<a href="https://colab.research.google.com/github/Auta01/Hugging-face/blob/main/Text2vediogen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

#load the model
pipe = StableDiffusionPipeline.from_pretrained('runwayml/stable-diffusion-v1-5', torch_dtype=torch.float16)
pipe = pipe.to('cuda')

In [ ]:
prompt = 'A man drinking water'
frames = []

for i in range(10):
  # Get the generated image. The output of pipe() has an 'images' attribute (a list).
  # We assume we want the first image from this list.
  generated_image = pipe(prompt).images[0]
  frames.append(generated_image) # Append the generated image to the list

In [ ]:
import cv2
import numpy as np
from PIL import Image # Import Image for type checking

# Save frames as images and stitch into a video

# The 'frames' variable is expected to be a list of PIL Image objects.
# If it's a single Image object, wrap it in a list to allow processing.
# This handles cases where only one image is generated or the list structure is lost.
if isinstance(frames, Image.Image):
    print("Warning: 'frames' was a single Image object. Wrapping it in a list for processing.")
    frames_to_process = [frames]
elif isinstance(frames, list):
    frames_to_process = frames
else:
    print("Error: 'frames' variable is of an unexpected type. Expected PIL Image or list of PIL Images.")
    raise TypeError("Expected PIL Image or list of PIL Images for 'frames' variable.")


if frames_to_process: # Ensure frames list is not empty
    # Get dimensions from the first frame
    # frames_to_process contains PIL Image objects, so .size gives (width, height)
    width, height = frames_to_process[0].size
    frame_size = (width, height) # OpenCV expects (width, height) tuple

    # Define video parameters
    fps = 5 # Frames per second for the output video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for MP4 video (e.g., 'mp4v', 'XVID', 'MJPG')

    # Create VideoWriter object
    out = cv2.VideoWriter('output_video.mp4', fourcc, fps, frame_size)

    for i, frame_pil in enumerate(frames_to_process):
        # Convert PIL Image (RGB) to NumPy array (for OpenCV)
        # OpenCV typically expects BGR, so convert from RGB
        frame_np = np.array(frame_pil)
        frame_cv = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)

        # Save individual frame as an image file
        cv2.imwrite(f'frame_{i:03d}.png', frame_cv) # Using f-string for padded index (e.g., frame_000.png)

        # Write the converted frame to the video file
        out.write(frame_cv)

    # Release the VideoWriter object once all frames are written
    out.release()
    print("Video 'output_video.mp4' created and individual frames saved.")
else:
    print("The 'frames_to_process' list is empty. Please ensure images are generated in the previous step.")